# Dataset Preprocessing

Transform the raw signals (EEG, ET, HR) into a clean dataset, ready to use for modeling.

**Pipeline:**
1. Eye Tracking: Extract 4 key features and calculate mean/std per meme.
2. EEG: z-score per subject, then features per channel and band, then extra features (differences, ratios, globals)
3. Expansion of Task21, Task22, Task23.


**Output:**
- Train: `data/processed/train_model_ready.parquet`
- Test: `data/processed/test_model_ready.parquet`


In [42]:
#Libraries
import os
import ast
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
tqdm.pandas()


#os.chdir("C:/Users/diego/Desktop/Master Neuro/M2/Internship_Valencia/multimodal-exist")
os.chdir("/home/diegoz/projects/multimodal-exist")

In [43]:
#Paths
OUTPUT_DIR = os.path.join("data", "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

#Load Dataset
train_file = os.path.join(OUTPUT_DIR, "train_base.parquet")
test_file = os.path.join(OUTPUT_DIR, "test_base.parquet")

df_train = pd.read_parquet(train_file)
#df_test = pd.read_parquet(test_file)

print(f"  Train: {df_train.shape[0]} samples")
#print(f"  Test:  {df_test.shape[0]} samples")

  Train: 3984 samples


In [44]:
df_train.head(3)

,id,lang,text,image_file,split,ET_raw,HR_raw,EEG_raw,task21_hard,task21_valid_hard,task21_soft,task22_hard,task22_valid_hard,task22_soft,task23_hard,task23_valid_hard,task23_soft,task23_num_hard_labels
0,110887,es,A VECES QUISIERAIR/ALZUMBA www.facebook.com/Oi...,110887.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1,True,1.000000,1.0,True,0.833333,"{'IDEOLOGICAL-INEQUALITY': 0, 'MISOGYNY-NON-SE...",True,{'IDEOLOGICAL-INEQUALITY': 0.16666666666666666...,2
1,110466,es,Se necesita cuidadora para adulto mayor.... fo...,110466.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1,True,0.500000,1.0,True,1.000000,"{'IDEOLOGICAL-INEQUALITY': 0, 'MISOGYNY-NON-SE...",True,{'IDEOLOGICAL-INEQUALITY': 0.16666666666666666...,2
2,111269,es,tomboy como son el anime y manga pre to tomboy...,111269.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1,True,0.833333,1.0,True,1.000000,"{'IDEOLOGICAL-INEQUALITY': 0, 'MISOGYNY-NON-SE...",True,"{'IDEOLOGICAL-INEQUALITY': 0.0, 'MISOGYNY-NON-...",1


## 1. Eye Tracking Aggregation

Selected Features (based on cognitive relevance - Sensorial Analysis)
- `reaction_time` (ms to s).
- `fixations_count`.
- `fixations_duration_mean_ns` (ns to ms).
- `saccades_count`.

Per meme we extract the mean and the std of valid subjects.

In [45]:
ET_FEATURE_MAP = {
    "reaction_time":"et_reaction_time",
    "fixations_count":"et_fixations",
    "fixations_duration_mean_ns":"et_fixation_duration",
    "saccades_count":"et_saccades"}

In [46]:
def aggregate_et_features(et_raw):

    result = {"et_n_users": 0}
    for feat_name in ET_FEATURE_MAP.values():
        result[f"{feat_name}_mean"] = np.nan
        result[f"{feat_name}_std"]  = np.nan

    if not isinstance(et_raw, dict):
        return result

    valid_users = {u: f for u, f in et_raw.items() if isinstance(f, dict)}
    result["et_n_users"] = len(valid_users)

    if len(valid_users) == 0:
        return result


    collected = {k: [] for k in ET_FEATURE_MAP}
    for _, feats in valid_users.items():
        for raw_key in ET_FEATURE_MAP:
            v = feats.get(raw_key, None)
            if v is not None:
                collected[raw_key].append(v)

    collected["reaction_time"]             = [x / 1_000     for x in collected["reaction_time"]]
    collected["fixations_duration_mean_ns"] = [x / 1_000_000 for x in collected["fixations_duration_mean_ns"]]

    for raw_key, feat_name in ET_FEATURE_MAP.items():
        vals = pd.to_numeric(pd.Series(collected[raw_key]), errors="coerce").dropna()
        if len(vals) > 0:
            result[f"{feat_name}_mean"] = vals.mean()
            result[f"{feat_name}_std"]  = vals.std() if len(vals) > 1 else 0.0

    return result

In [47]:
for df, name in [(df_train, "train")]:
    et_agg = df["ET_raw"].apply(aggregate_et_features).apply(pd.Series)
    et_cols_to_add = [c for c in et_agg.columns if c not in df.columns]
    df[et_agg.columns] = et_agg.values

et_feature_cols = [c for c in df_train.columns if c.startswith("et_")]
print(f"\nET columns ({len(et_feature_cols)}): {et_feature_cols}")


ET columns (9): ['et_n_users', 'et_reaction_time_mean', 'et_reaction_time_std', 'et_fixations_mean', 'et_fixations_std', 'et_fixation_duration_mean', 'et_fixation_duration_std', 'et_saccades_mean', 'et_saccades_std']


## 2. EEG Aggregation

1. Z-score per subject
2. Raw Features per channel (16 channels * 5 bands = 80 raw)
3. Global Features (mean between channels)
4. Extra Features (Diferences ('`Theta − Alpha`,...), Ratio, )
5. We agregate per subject: mean and std per meme


In [48]:
# EEG Constants
BANDS = ["Alpha", "Beta", "Theta", "Gamma", "Delta"]
N_CH  = 16
AUX   = 1e-6   

#Mapping
REGION_MAP = {
    "frontal_polar": [0, 1],         # Fp1, Fp2
    "frontal":       [8, 9, 10, 11], # F7, F8, F3, F4
    "central":       [2, 3],         # C3, C4
    "temporal":      [12, 13],       # T7, T8
    "parietal":      [4, 5, 14, 15], # P7, P8, P3, P4
    "occipital":     [6, 7],         # O1, O2
}

LEFT_CHANNELS  = [0, 2, 4, 6, 8, 10, 12, 14]  # Fp1, C3, P7, O1, F7, F3, T7, P3
RIGHT_CHANNELS = [1, 3, 5, 7, 9, 11, 13, 15]  # Fp2, C4, P8, O2, F8, F4, T8, P4

#Feature that we are going to include base on our statictical analysis

#Log-Ratios 
LOG_RATIO_PAIRS = [
    ("Beta",  "Alpha"),
    ("Theta", "Beta"),
    ("Theta", "Alpha"),
    ("Gamma", "Beta"),
    ("Gamma", "Alpha")]

#Global Differences
GLOBAL_DIFF_PAIRS = [
    ("Alpha", "Delta"),   # Task 2.1, SEXUAL-VIOLENCE
    ("Alpha", "Theta"),   # Task 2.1, Task 2.2, SEXUAL-VIOLENCE
    ("Alpha", "Beta"),
    ("Beta",  "Gamma"),   # Task 2.1
    ("Beta",  "Delta"),
    ("Beta",  "Theta"),
    ("Delta", "Gamma"),
    ("Delta", "Theta"),   # SEXUAL-VIOLENCE
    ("Gamma", "Theta"),]

#Regional differences
REGIONAL_DIFF_PAIRS = [
    ("Alpha", "Theta",  "frontal_polar"),# frontal_polar — Task 2.2 y Task 2.3 IDEOLOGICAL-INEQUALITY
    ("Alpha", "Beta",   "frontal_polar"),
    ("Alpha", "Gamma",  "frontal_polar"),
    ("Beta",  "Delta",  "frontal_polar"),
    ("Beta",  "Theta",  "frontal_polar"),
    ("Delta", "Gamma",  "frontal_polar"),
    ("Gamma", "Theta",  "frontal_polar"),# central — Task 2.1 y Task 2.3 SEXUAL-VIOLENCE
    ("Alpha", "Gamma",  "central"),
    ("Alpha", "Theta",  "central"),
    ("Beta",  "Gamma",  "central"),
    ("Beta",  "Delta",  "central"),
    ("Delta", "Gamma",  "central"),
    ("Delta", "Theta",  "central"),# temporal — Task 2.3 SEXUAL-VIOLENCE
    ("Beta",  "Delta",  "temporal"),
    ("Delta", "Theta",  "temporal"),# parietal — Task 2.3 STEREOTYPING-DOMINANCE
    ("Alpha", "Beta",   "parietal"),# occipital — Task 2.3 STEREOTYPING-DOMINANCE
    ("Beta",  "Theta",  "occipital"),
    ("Gamma", "Theta",  "occipital"),# frontal — Task 2.2
    ("Alpha", "Theta",  "frontal"),
]

print(f"  Bands                       : {BANDS}")
print(f"  Regions                     : {list(REGION_MAP.keys())}")
print(f"  Log-ratio pairs             : {len(LOG_RATIO_PAIRS)} × 6 regions = {len(LOG_RATIO_PAIRS)*6}")
print(f"  Global diff pairs selected  : {len(GLOBAL_DIFF_PAIRS)}")
print(f"  Regional diff pairs selected: {len(REGIONAL_DIFF_PAIRS)}")

  Bands                       : ['Alpha', 'Beta', 'Theta', 'Gamma', 'Delta']
  Regions                     : ['frontal_polar', 'frontal', 'central', 'temporal', 'parietal', 'occipital']
  Log-ratio pairs             : 5 × 6 regions = 30
  Global diff pairs selected  : 9
  Regional diff pairs selected: 19


In [49]:
#Function to standardize the EEG features of each user (z-scoring)
def zscore_user_eeg(user_feats):
    keys = [f"EXG_Channel_{ch}_{band}_power"
            for ch in range(N_CH) for band in BANDS]
    vals = np.array([user_feats.get(k, np.nan) for k in keys], dtype=float)

    mask = ~np.isnan(vals)
    if mask.sum() > 1:
        mu, sigma = vals[mask].mean(), vals[mask].std()
        if sigma > 0:
            vals[mask] = (vals[mask] - mu) / sigma

    return dict(zip(keys, vals))


#Function to create EEG Features (raw, global, regional, lateralization, diferencias, log-ratios)
def extract_eeg_user_features(d_zscored, d_raw):
    """
    d_zscored: dict con valores z-scored (para raw features, globales, regionales, diferencias)
    d_raw: dict con valores originales (para log-ratios)
    """

    feats = {}
    d = d_zscored  

    #Raw Band Power per channel (z-scored)
    for ch in range(N_CH):
        for band in BANDS:
            key = f"EXG_Channel_{ch}_{band}_power"
            feats[f"EEG_{key}"] = d.get(key, np.nan)

    #Global, Regionales, and Lateralization features (z-scored)
    for band in BANDS:
        vals = [d.get(f"EXG_Channel_{ch}_{band}_power", np.nan) for ch in range(N_CH)]
        vals = [v for v in vals if not np.isnan(v)]
        feats[f"EEG_{band}_global"] = np.mean(vals) if vals else np.nan

    for region, ch_list in REGION_MAP.items():
        for band in BANDS:
            vals = [d.get(f"EXG_Channel_{ch}_{band}_power", np.nan) for ch in ch_list]
            vals = [v for v in vals if not np.isnan(v)]
            feats[f"EEG_{band}_{region}"] = np.mean(vals) if vals else np.nan

    for band in BANDS:
        l_vals = [d.get(f"EXG_Channel_{ch}_{band}_power", np.nan) for ch in LEFT_CHANNELS]
        r_vals = [d.get(f"EXG_Channel_{ch}_{band}_power", np.nan) for ch in RIGHT_CHANNELS]
        l_vals = [v for v in l_vals if not np.isnan(v)]
        r_vals = [v for v in r_vals if not np.isnan(v)]
        feats[f"EEG_{band}_left"]  = np.mean(l_vals) if l_vals else np.nan
        feats[f"EEG_{band}_right"] = np.mean(r_vals) if r_vals else np.nan

    for b1, b2 in GLOBAL_DIFF_PAIRS:
        v1 = feats.get(f"EEG_{b1}_global", np.nan)
        v2 = feats.get(f"EEG_{b2}_global", np.nan)
        feats[f"EEG_{b1}_minus_{b2}_global"] = (v1 - v2 if not (np.isnan(v1) or np.isnan(v2)) else np.nan)

    for b1, b2, region in REGIONAL_DIFF_PAIRS:
        v1 = feats.get(f"EEG_{b1}_{region}", np.nan)
        v2 = feats.get(f"EEG_{b2}_{region}", np.nan)
        feats[f"EEG_{b1}_minus_{b2}_{region}"] = (v1 - v2 if not (np.isnan(v1) or np.isnan(v2)) else np.nan)

    #Log-Ratios, we use d_raw because log-ratios should be computed on original power values (non z-scored), 
    # which are always >= 0. We add a small constant (AUX) to avoid log(0) issues.
    for region, ch_list in REGION_MAP.items():
        for num, den in LOG_RATIO_PAIRS:
            raw_num = [d_raw.get(f"EXG_Channel_{ch}_{num}_power", np.nan) for ch in ch_list]
            raw_den = [d_raw.get(f"EXG_Channel_{ch}_{den}_power", np.nan) for ch in ch_list]
            raw_num = [v for v in raw_num if not np.isnan(v) and v >= 0]
            raw_den = [v for v in raw_den if not np.isnan(v) and v >= 0]

            if raw_num and raw_den:
                v_num = np.mean(raw_num)
                v_den = np.mean(raw_den)
                feats[f"EEG_log_{num}_{den}_ratio_{region}"] = np.log((v_num + AUX) / (v_den + AUX))
            else:
                feats[f"EEG_log_{num}_{den}_ratio_{region}"] = np.nan
    return feats

# Renaming the features to be more clear and consistent 
_dummy_d = {f"EXG_Channel_{ch}_{band}_power": 0.0 for ch in range(N_CH) for band in BANDS}
_dummy_feats    = extract_eeg_user_features(_dummy_d, d_raw=_dummy_d)
EEG_FEATURE_NAMES = list(_dummy_feats.keys())

print(f"  Total features por sujeto  : {len(EEG_FEATURE_NAMES)}")
print(f"  Raw por canal (16*5)        : 80")
print(f"  Globales (5 bandas)         : 5")
print(f"  Regionales (6 reg * 5 band) : 30")
print(f"  Left / Right (5 bandas ×2)  : 10")
print(f"  Dif. globales seleccionadas : {len(GLOBAL_DIFF_PAIRS)}")
print(f"  Dif. regionales seleccionadas: {len(REGIONAL_DIFF_PAIRS)}")
print(f"  Log-ratios (6 reg * 5 pares): {len(REGION_MAP)*len(LOG_RATIO_PAIRS)}")

  Total features por sujeto  : 183
  Raw por canal (16*5)        : 80
  Globales (5 bandas)         : 5
  Regionales (6 reg * 5 band) : 30
  Left / Right (5 bandas ×2)  : 10
  Dif. globales seleccionadas : 9
  Dif. regionales seleccionadas: 19
  Log-ratios (6 reg * 5 pares): 30


In [50]:
# Function to compute the aggregated EEG features for a meme, based on the raw EEG data.
def aggregate_eeg_features(eeg_raw):
    result = {"EEG_n_users": 0}
    for feat in EEG_FEATURE_NAMES:
        result[feat] = np.nan

    if not isinstance(eeg_raw, dict):
        return result

    valid_users = {u: f for u, f in eeg_raw.items() if isinstance(f, dict)}
    result["EEG_n_users"] = len(valid_users)
    if len(valid_users) == 0:
        return result

    user_rows = []
    for _, feats in valid_users.items():
        d_zscored = zscore_user_eeg(feats)
        user_rows.append(extract_eeg_user_features(d_zscored=d_zscored, d_raw=feats))

    user_df = pd.DataFrame(user_rows)
    for feat in EEG_FEATURE_NAMES:
        if feat in user_df.columns:
            vals = pd.to_numeric(user_df[feat], errors="coerce").dropna()
            if len(vals) > 0:
                result[feat] = float(vals.mean())

    return result

print(f"Output per meme: {len(EEG_FEATURE_NAMES) + 1} columns (features + eeg_n_users)")

Output per meme: 184 columns (features + eeg_n_users)


In [51]:
for df, name in [(df_train, "train")]:
    print(f"\nProcesando EEG — {name} ({len(df)} memes)...")
    eeg_agg = df["EEG_raw"].progress_apply(aggregate_eeg_features).apply(pd.Series)
    df[eeg_agg.columns] = eeg_agg.values
    
EEG_COLS = ["EEG_n_users"] + EEG_FEATURE_NAMES
print(f"\nTotal EEG columns: {len(EEG_COLS)}")


Procesando EEG — train (3984 memes)...


  0%|          | 0/3984 [00:00<?, ?it/s]


Total EEG columns: 184


/tmp/ipykernel_509227/106741134.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[eeg_agg.columns] = eeg_agg.values
/tmp/ipykernel_509227/106741134.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[eeg_agg.columns] = eeg_agg.values
/tmp/ipykernel_509227/106741134.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `ne

## 3. Labels

In [52]:
#Re name labels for better consistency
rename_map = {
    "task21_valid_hard": "task21_valid",
    "task22_valid_hard": "task22_valid",
    "task23_valid_hard": "task23_valid",
    "task21_hard": "sexism",
    "task22_hard": "direct"}

df_train.rename(columns=rename_map, inplace=True)
#df_test.rename(columns=rename_map,  inplace=True)



In [53]:
#Task 2.1: Sexism Detection (binary)

print("Task 2.1 — Distribución (Train):")
print("Task 2.1. Train Distribution:")
print(df_train["sexism"].value_counts().rename({0: "Non-Sexist", 1: "Sexist"}))
print(f"\nBalance: {df_train['sexism'].mean():.1%} sexists")

#if "sexism" in df_test.columns:
#    print("\nTask 2.1 — Distribución (Test):")
#    print(df_test["sexism"].value_counts().rename({0: "Non-Sexist", 1: "Sexist"}))

Task 2.1 — Distribución (Train):
Task 2.1. Train Distribution:
Sexist        2617
Non-Sexist    1367
Name: sexism, dtype: int64

Balance: 65.7% sexists


In [54]:
#Task 2.2: Source Intention (Binary - Direct or Judgemental) 
print("Task 2.2 — Distribución (Train, solo sexistas):")
t22_train = df_train[df_train["sexism"] == 1]["direct"]
print(t22_train.value_counts().rename({1: "DIRECT", 0: "JUDGEMENTAL"}))
print(f"  NaN (no aplica): {t22_train.isna().sum()}")

#if "task22_hard" in df_test.columns:
#    print("\nTask 2.2 — Distribución (Test, solo sexistas):")
#    t22_test = df_test[df_test["task21_hard"] == 1]["task22_hard"]
#    print(t22_test.value_counts().rename({1: "DIRECT", 0: "JUDGEMENTAL"}))

Task 2.2 — Distribución (Train, solo sexistas):
DIRECT         2005
JUDGEMENTAL     612
Name: direct, dtype: int64
  NaN (no aplica): 0


In [55]:
# Task 2.3 Sexism Categorization (Multilabel - 5 categories)
TASK23_CATEGORIES = [
    "IDEOLOGICAL-INEQUALITY",
    "STEREOTYPING-DOMINANCE",
    "OBJECTIFICATION",
    "SEXUAL-VIOLENCE",
    "MISOGYNY-NON-SEXUAL-VIOLENCE"]

TASK23_RENAME = {
    "IDEOLOGICAL-INEQUALITY":        "ideological_inequality",
    "STEREOTYPING-DOMINANCE":         "stereotyping_dominance",
    "OBJECTIFICATION":                "objectification",
    "SEXUAL-VIOLENCE":                "sexual_violence",
    "MISOGYNY-NON-SEXUAL-VIOLENCE":   "misogyny_non_sexual_violence"
}


def expand_task23(df, col, prefix):
    expanded = df[col].apply(lambda x: x if isinstance(x, dict) else {cat: np.nan for cat in TASK23_CATEGORIES}).apply(pd.Series)
    expanded = expanded.rename(columns=TASK23_RENAME)
    expanded.columns = [f"{prefix}{c}" for c in expanded.columns]
    return expanded


for df, name in [(df_train, "train")]:
    # Hard labels
    t23_hard = expand_task23(df, "task23_hard", "CAT_")
    df[t23_hard.columns] = t23_hard.values
    df.drop(columns=["task23_hard"], inplace=True, errors="ignore")
    print(f"Categories Expanded")

print("\nTask 2.3 Category Distribution (Train, only sexists):")
t23_cols_hard = [c for c in df_train.columns if c.startswith("CAT_")]
sexist_mask = df_train["sexism"] == 1
print(df_train.loc[sexist_mask, t23_cols_hard].sum().sort_values(ascending=False))

Categories Expanded

Task 2.3 Category Distribution (Train, only sexists):
CAT_stereotyping_dominance          1186
CAT_objectification                 1111
CAT_ideological_inequality           988
CAT_sexual_violence                  539
CAT_misogyny_non_sexual_violence     433
dtype: int64


## 4. Columns Selection

In [56]:
meta_cols = ["id", "lang", "text", "image_file", "split"]

et_cols_final  = sorted([c for c in df_train.columns if c.startswith("et_")])
eeg_cols_final = sorted([c for c in df_train.columns if c.startswith("EEG_")])

#We are not going to include the HR features in the model but we will keep it for future analysis.
hr_meta_cols = [c for c in df_train.columns if c.startswith("hr_") or c == "HR_raw"]


task21_cols = ["task21_valid", "sexism"]
task22_cols = ["task22_valid", "direct"]

task23_hard_cols = sorted([c for c in df_train.columns if c.startswith("CAT_")])
task23_cols = ["task23_valid", "task23_num_hard_labels"] + task23_hard_cols 

# Columns
final_cols = (
    meta_cols +
    et_cols_final +
    eeg_cols_final +
    task21_cols +
    task22_cols +
    task23_cols)

print(f"Columnas finales: {len(final_cols)}")
print(f"  Meta:       {len(meta_cols)}")
print(f"  ET:         {len(et_cols_final)}")
print(f"  EEG:        {len(eeg_cols_final)}")
print(f"  Task21:     {len([c for c in task21_cols if c in df_train.columns])}")
print(f"  Task22:     {len([c for c in task22_cols if c in df_train.columns])}")
print(f"  Task23:     {len([c for c in task23_cols if c in df_train.columns])}")

df_train = df_train[final_cols].copy()
#df_test  = df_test[final_cols].copy()

#Remove Raw columns
df_train.drop(columns=["ET_raw", "EEG_raw"], inplace=True, errors="ignore")
df_train.drop("EEG_n_users", axis=1, inplace=True, errors="ignore")

print(f"Train final: {df_train.shape}")
#print(f"Test  final: {df_test.shape}")

df_train.head(5)

Columnas finales: 210
  Meta:       5
  ET:         9
  EEG:        185
  Task21:     2
  Task22:     2
  Task23:     7
Train final: (3984, 208)


,id,lang,text,image_file,split,et_fixation_duration_mean,et_fixation_duration_std,et_fixations_mean,et_fixations_std,et_n_users,...,sexism,task22_valid,direct,task23_valid,task23_num_hard_labels,CAT_ideological_inequality,CAT_misogyny_non_sexual_violence,CAT_objectification,CAT_sexual_violence,CAT_stereotyping_dominance
0,110887,es,A VECES QUISIERAIR/ALZUMBA www.facebook.com/Oi...,110887.jpeg,Train,290.553894,46.333007,42.92820,19.900530,2.0,...,1,True,1.0,True,2,0,1,1,0,0
1,110466,es,Se necesita cuidadora para adulto mayor.... fo...,110466.jpeg,Train,272.211591,32.506004,48.00000,25.238859,3.0,...,1,True,1.0,True,2,0,0,0,1,1
2,111269,es,tomboy como son el anime y manga pre to tomboy...,111269.jpeg,Train,259.975251,129.980125,40.33335,3.771213,2.0,...,1,True,1.0,True,1,0,0,1,0,0
3,110593,es,HOY QUIERO FELICITAR A TODAS LAS MUJERES DE ES...,110593.jpeg,Train,303.698083,8.588806,31.90805,0.130037,2.0,...,0,True,0.0,False,0,0,0,0,0,0
4,110946,es,DUCHATE GUARRA GRACIAS memegenerator.es,110946.jpeg,Train,228.190844,132.391664,19.00000,1.414214,2.0,...,1,True,1.0,True,1,0,0,1,0,0


In [57]:
df_train[(df_train["CAT_ideological_inequality"]==0) &
    (df_train["CAT_stereotyping_dominance"]==0) &
    (df_train["CAT_objectification"]==0) &
    (df_train["CAT_sexual_violence"]==0) &
    (df_train["CAT_misogyny_non_sexual_violence"]==0)&
    (df_train["sexism"]==1)]

,id,lang,text,image_file,split,et_fixation_duration_mean,et_fixation_duration_std,et_fixations_mean,et_fixations_std,et_n_users,...,sexism,task22_valid,direct,task23_valid,task23_num_hard_labels,CAT_ideological_inequality,CAT_misogyny_non_sexual_violence,CAT_objectification,CAT_sexual_violence,CAT_stereotyping_dominance
75,111505,es,FRIENDZONE LEVEL 99 PAGAFANTAS Nivel: 99 más e...,111505.jpeg,Train,211.213178,156.024987,17.500000,2.121320,2.0,...,1,True,1.0,False,0,0,0,0,0,0
105,111857,es,Movistar may_rondon от 941 Me gusta ... Instag...,111857.jpeg,Train,341.046235,3.196800,41.500000,7.778175,2.0,...,1,True,1.0,False,0,0,0,0,0,0
108,111502,es,¿QUÉ HARÍAS SI TU EX TE DIERA ESTO? Nada mas u...,111502.jpeg,Train,287.460134,4.408947,116.500000,71.417785,2.0,...,1,True,1.0,False,0,0,0,0,0,0
154,111499,es,"og rep BLITZ Jason Statham 22:30 Javier, 51 ti...",111499.jpeg,Train,184.011919,90.383486,25.000000,19.798990,2.0,...,1,True,1.0,False,0,0,0,0,0,0
161,111121,es,ii For No se felicita el día de la mujer se co...,111121.jpeg,Train,269.923030,18.401187,58.544000,48.728143,2.0,...,1,True,0.0,False,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3797,211889,en,Him: I hope you ain't crazy Me: KAUP,211889.jpeg,Train,359.777380,205.884495,37.333333,28.448784,3.0,...,1,True,0.0,False,0,0,0,0,0,0
3871,211663,en,Average step sister stuck in the washing machi...,211663.jpeg,Train,287.468423,92.784983,41.000000,6.000000,3.0,...,1,True,1.0,False,0,0,0,0,0,0
3878,211996,en,I just think there's some things women shouldn...,211996.jpeg,Train,304.254372,26.531512,59.000000,21.213203,2.0,...,1,True,0.0,False,0,0,0,0,0,0
3884,211416,en,MY SEXUALITY & GENDER IDENTITY AREN'T THE PROB...,211416.jpeg,Train,265.139145,32.517343,77.000000,42.426407,2.0,...,1,True,0.0,False,0,0,0,0,0,0


In [58]:
df_train.columns

Index(['id', 'lang', 'text', 'image_file', 'split',
       'et_fixation_duration_mean', 'et_fixation_duration_std',
       'et_fixations_mean', 'et_fixations_std', 'et_n_users',
       ...
       'sexism', 'task22_valid', 'direct', 'task23_valid',
       'task23_num_hard_labels', 'CAT_ideological_inequality',
       'CAT_misogyny_non_sexual_violence', 'CAT_objectification',
       'CAT_sexual_violence', 'CAT_stereotyping_dominance'],
      dtype='object', length=208)

In [59]:
#Task 2.1
print(f"\nTask 2.1 (sexism):")
print(df["sexism"].value_counts().rename({0: 'Non-Sexist', 1: 'Sexist'}))

#Extracting physio columns (EEG and ET features)
EEG_COLS = sorted([
    c for c in df.columns
    if c.startswith("EEG_") and c not in {"EEG_n_users", "EEG_raw", "et_n_users"}])

ET_COLS = sorted([
    c for c in df.columns
    if c.startswith("et_") and c not in {"EEG_n_users", "EEG_raw", "et_n_users"}]
    )
PHYSIO_COLS = EEG_COLS + ET_COLS
PHYSIO_DIM  = len(PHYSIO_COLS)

print(f"\nEEG features : {len(EEG_COLS)}")
print(f"ET  features : {len(ET_COLS)}")
print(f"Total physio : {PHYSIO_DIM}")


Task 2.1 (sexism):
Sexist        2617
Non-Sexist    1367
Name: sexism, dtype: int64

EEG features : 183
ET  features : 8
Total physio : 191


In [60]:
# ── DIAGNÓSTICO PHYSIO ─────────────────────────────────────────────
print(f"Total filas: {len(df_train)}")
print(f"Total columnas physio: {len(PHYSIO_COLS)}")

nan_por_col = df_train[PHYSIO_COLS].isna().sum()
print(f"\nColumnas physio con NaN: {(nan_por_col > 0).sum()} / {len(PHYSIO_COLS)}")
print(nan_por_col[nan_por_col > 0])

filas_con_nan = df_train[PHYSIO_COLS].isna().any(axis=1).sum()
print(f"\nFilas con al menos 1 NaN en physio: {filas_con_nan} ({filas_con_nan/len(df_train)*100:.1f}%)")

print(f"\nEstadísticas básicas (antes de imputar):")
print(df_train[PHYSIO_COLS].describe().loc[['min','max','mean','std']].T.head(10))

Total filas: 3984
Total columnas physio: 191

Columnas physio con NaN: 30 / 191
EEG_log_Beta_Alpha_ratio_central            775
EEG_log_Beta_Alpha_ratio_frontal            235
EEG_log_Beta_Alpha_ratio_frontal_polar     1032
EEG_log_Beta_Alpha_ratio_occipital          759
EEG_log_Beta_Alpha_ratio_parietal           191
EEG_log_Beta_Alpha_ratio_temporal           691
EEG_log_Gamma_Alpha_ratio_central           878
EEG_log_Gamma_Alpha_ratio_frontal           305
EEG_log_Gamma_Alpha_ratio_frontal_polar    1231
EEG_log_Gamma_Alpha_ratio_occipital         801
EEG_log_Gamma_Alpha_ratio_parietal          121
EEG_log_Gamma_Alpha_ratio_temporal          710
EEG_log_Gamma_Beta_ratio_central            664
EEG_log_Gamma_Beta_ratio_frontal            176
EEG_log_Gamma_Beta_ratio_frontal_polar      751
EEG_log_Gamma_Beta_ratio_occipital          585
EEG_log_Gamma_Beta_ratio_parietal           179
EEG_log_Gamma_Beta_ratio_temporal           635
EEG_log_Theta_Alpha_ratio_central           830
EEG_log_

In [61]:
df_train[df_train["EEG_log_Beta_Alpha_ratio_central"].isna()][["EEG_Alpha_central","EEG_Beta_central","EEG_log_Beta_Alpha_ratio_central"]].head(10)

,EEG_Alpha_central,EEG_Beta_central,EEG_log_Beta_Alpha_ratio_central
5,-0.399247,-0.167450,NaN
10,-0.241955,0.169500,NaN
19,-0.185200,0.124960,NaN
23,-0.383584,0.072637,NaN
36,-0.883486,-0.523556,NaN
37,-1.222642,-0.985889,NaN
45,-0.816425,-0.626361,NaN
49,0.098644,-0.119167,NaN
59,-0.907430,-0.557811,NaN
63,0.116773,-0.064276,NaN


In [39]:
np.log((-0.167450 + AUX) / (-0.399247 + AUX))

-0.8688989409692497

In [29]:
#Export
train_out = os.path.join(OUTPUT_DIR, "train_model_ready.parquet")
test_out  = os.path.join(OUTPUT_DIR, "test_model_ready.parquet")

df_train.to_parquet(train_out, index=False)
#df_test.to_parquet(test_out,  index=False)

print(f"Train saved: {train_out}")
print(f"Test saved: {test_out}")

Train saved: data/processed/train_model_ready.parquet
Test saved: data/processed/test_model_ready.parquet
